# SQL and Parquet

- Parquet files are perfect as a backing data store for SQL queries in Spark. While it is possible to run the same queries directly via Spark's Python functions, sometimes it's easier to run SQL queries alongside the Python options.

- For this example, we're going to read in the Parquet file we created in the last exercise and register it as a SQL table. Once registered, we'll run a quick query against the table (aka, the Parquet file).

- The `spark` object and the `AA_DFW_ALL.parquet` file are available for you automatically.

## Instructions

- Import the `AA_DFW_ALL.parquet` file into `flights_df`.
- Use the `createOrReplaceTempView` method to alias the `flights` table.
- Run a Spark SQL query against the `flights` table.

In [1]:
# Explicitly Point Spark to the Correct Python Executable
# If you already have numpy installed in a specific environment (like conda or a virtualenv), but PySpark is defaulting to /usr/bin/python3 or system Python, 
# explicitly set the Python environment variables before initializing SparkContext:

import os
import sys

# Set these before initializing SparkContext (sc)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession

In [3]:
# If you are developing an application you should use local mode. 
# Running in yarn mode is bit time consuming than running in local mode
# If you still insist to run in yarn mode then make sure that
# 1. You assigned more than one processor in the VM settings
# 2. Run only one kernel at a time

In [4]:
# spark = SparkSession. \
#     builder. \
#     enableHiveSupport(). \
#     appName('Demo'). \
#     master('yarn'). \
#     getOrCreate()

In [5]:
spark = SparkSession. \
    builder. \
    enableHiveSupport(). \
    appName('Demo'). \
    master('local[*]'). \
    getOrCreate()

In [6]:
sc = spark.sparkContext

In [7]:
# Read the Parquet file into flights_df
flights_df = spark.read.parquet('AA_DFW_ALL.parquet')
flights_df = flights_df.withColumn("flight_duration", flights_df.flight_duration.cast('double'))

# Register the temp table
flights_df.createOrReplaceTempView('flights')

# Run a SQL query of the average flight duration
avg_duration = spark.sql('SELECT avg(flight_duration) from flights').collect()[0]
print('The average flight time is: %f' % avg_duration)

The average flight time is: 151.327621


# Understanding Parquet file importance and advantage over csv file format

In [ ]:
# Look in the solution file of EX_2 IN Lab-11 of spark
people1_df_sub = people1_df.select('name', 'sex')

people1_df_sub.write.format('csv').save('people1_csv')
# HERE he data folder people1_csv will be created on hdfs and inside it we will have data stored in no of files = no of columns we selected above
# CSV file format has a problem it doesnt store schema we have to create a intermediate table schema like 'pc' here

pc = StructType([
    StructField('name', StringType(), True),
    StructField('sex', StringType(), True),
])

df = spart.read.format('csv').option("header","true").load(path = 'people1_csv', schema = pc)

people1_df.show()

In [ ]:
people1_df_sub = people1_df.select('name', 'sex')

people1_df_sub.write.format('parquet').save('people1_parquet') # run it only once and commentout it when running this script second time



df = spark.read.format('parquet').load('people1_parquet')
df.groupBy('sex').count().show()
df.show()
                                            
#people1_df.show()

In [ ]:
%%sh 
hdfs dfs -ls